## 1. Problem Formulation

| Component | Definition |
|---|---|
| **State** | Current town the trailer is at, drawn from {Nairobi, Naivasha, Narok, Nakuru, Bomet, Kericho, Eldoret, Kisumu} |
| **Initial State** | `Nairobi` |
| **Goal State** | `Kisumu` |
| **Actions** | `Actions(s)` = set of towns with a direct directed highway edge from `s` |
| **Transition Model** | `Result(s, a) = a` (town reached by driving the highway edge) |
| **Path Cost** | `c(s, a, s')` = driving distance in km on that highway segment |
| **Solution** | Ordered sequence of towns from Nairobi to Kisumu |

**Search strategy:** Uniform Cost Search (UCS) — expands the frontier node with lowest `g(n)`, guaranteed optimal for non-negative edge costs.

In [1]:
# ---------------------------------------------------------------
# Directed graph:  node -> { successor: edge_cost (km) }
# Data taken directly from Part 1 of the Practical Exercise.
# ---------------------------------------------------------------
graph = {
    "Nairobi":  {"Naivasha": 90,  "Narok": 142},
    "Naivasha": {"Nakuru": 69},
    "Narok":    {"Bomet": 82},
    "Nakuru":   {"Kericho": 107,  "Eldoret": 154},
    "Bomet":    {"Kericho": 77},
    "Kericho":  {"Kisumu": 83},
    "Eldoret":  {"Kisumu": 117},
    "Kisumu":   {},                       # goal — no outgoing edges
}

START = "Nairobi"
GOAL  = "Kisumu"

# Quick sanity check
print("Nodes:", list(graph.keys()))
print("Total directed edges:",
      sum(len(v) for v in graph.values()))

Nodes: ['Nairobi', 'Naivasha', 'Narok', 'Nakuru', 'Bomet', 'Kericho', 'Eldoret', 'Kisumu']
Total directed edges: 9


In [2]:
class Node:
    """A node in the UCS search tree."""
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state     = state
        self.parent    = parent
        self.action    = action        # edge label that generated this node
        self.path_cost = path_cost     # g(n)

    def __lt__(self, other):           # heapq needs ordering
        return self.path_cost < other.path_cost

    def __repr__(self):
        return f"Node({self.state}, g={self.path_cost})"

    def path(self):
        """Walk parents to reconstruct the full path (root → this node)."""
        node, trail = self, []
        while node:
            trail.append(node)
            node = node.parent
        return list(reversed(trail))


def expand(graph, node):
    """Yield one child Node for every successor of node.state."""
    for successor, edge_cost in graph.get(node.state, {}).items():
        yield Node(state     = successor,
                   parent    = node,
                   action    = successor,
                   path_cost = node.path_cost + edge_cost)

In [3]:
import heapq

def uniform_cost_search(graph, start, goal, verbose=True):
    """
    Uniform Cost Search.
    Returns (goal_node, expansion_trace) or (None, trace) on failure.
    """
    root = Node(start)
    frontier = [root]                       # min-heap on g(n)
    reached  = {start: root}                # best g known per state
    trace    = []                           # (state, g) in expansion order
    step     = 0

    while frontier:
        node = heapq.heappop(frontier)      # pop lowest-cost node
        trace.append((node.state, node.path_cost))
        step += 1

        if verbose:
            fringe = sorted(((n.state, n.path_cost) for n in frontier),
                            key=lambda x: (x[1], x[0]))
            print(f"Step {step:>2}: expand {node.state:<10} g = {node.path_cost:<4} km"
                  f" | frontier = {fringe}")

        # ---- Goal test AFTER popping (correct UCS ordering) ----
        if node.state == goal:
            if verbose:
                print(f"\n✅ GOAL '{goal}' reached with total cost = {node.path_cost} km")
            return node, trace

        # ---- Expand successors ----
        for child in expand(graph, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                heapq.heappush(frontier, child)

    if verbose:
        print("\n❌ No path found.")
    return None, trace


# ---- Run the search ----
goal_node, trace = uniform_cost_search(graph, START, GOAL, verbose=True)

Step  1: expand Nairobi    g = 0    km | frontier = []
Step  2: expand Naivasha   g = 90   km | frontier = [('Narok', 142)]
Step  3: expand Narok      g = 142  km | frontier = [('Nakuru', 159)]
Step  4: expand Nakuru     g = 159  km | frontier = [('Bomet', 224)]
Step  5: expand Bomet      g = 224  km | frontier = [('Kericho', 266), ('Eldoret', 313)]
Step  6: expand Kericho    g = 266  km | frontier = [('Eldoret', 313)]
Step  7: expand Eldoret    g = 313  km | frontier = [('Kisumu', 349)]
Step  8: expand Kisumu     g = 349  km | frontier = []

✅ GOAL 'Kisumu' reached with total cost = 349 km
